# The dashboard (Phase 7, week 26)

Three numbers of the assistant, for each day:

- **The 95th-percentile latency of a request.** One request is one trace in Langfuse. Its latency is the
  latency of the root observation of the trace (for example `ask` for /ask and `approve` for /approve),
  the time from the question to the `done` event.
- **The cost of a request.** The sum of the costs of the model calls (the generations) of its trace,
  as your code sends them to Langfuse (Phase 3, week 8). The mean of the day.
- **The evaluation scores over time.** The scores of your result files under `evals/results/`, by the
  time of each run, and the share of good answers of the feedback of week 26 (the score `user-feedback`).

Langfuse shows the first number itself (Dashboards, "Langfuse Latency Dashboard", widget "P 95 Latency by
Use Case", line `ask`). It does not show the mean cost of a request, and the evaluation scores are not in
Langfuse. This notebook reads all three: the requests and the feedback from the public API of Langfuse, the
scores from the files. It writes nothing.

Open it with `make dashboard`, after `make up`. Run all cells again for new data.

In [ ]:
import json
import os
import re
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import httpx
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from labkit.capstone import percentile  # noqa: E402  (nearest rank, as in Phase 6)

DAYS = 14  # the days that the dashboard shows
STEPS = {"GENERATION", "TOOL", "RETRIEVER", "EMBEDDING"}  # a root of these types is one step, not a request
SINCE = datetime.now(timezone.utc) - timedelta(days=DAYS)
LANGFUSE = os.environ.get("LANGFUSE_BASE_URL", "http://localhost:3000").rstrip("/")
AUTH = (os.environ.get("LANGFUSE_PUBLIC_KEY", ""), os.environ.get("LANGFUSE_SECRET_KEY", ""))
RESULTS = ROOT / "evals" / "results"


def pages(path, **params):
    """All items of a list endpoint of the Langfuse public API, page by page (cursor)."""
    cursor = None
    while True:
        query = {**params, **({"cursor": cursor} if cursor else {})}
        response = httpx.get(f"{LANGFUSE}{path}", params=query, auth=AUTH, timeout=30)
        response.raise_for_status()
        page = response.json()
        yield from page["data"]
        cursor = (page.get("meta") or {}).get("cursor")
        if not cursor:
            return


def observations(fields, **params):
    return list(pages("/api/public/v2/observations", fromStartTime=SINCE.isoformat(),
                      fields=fields, limit=1000, **params))

## Latency and cost of each request

Each row of `requests` is one trace with a request: the day, the name of its root observation, the latency
in seconds and the cost in USD. The notebook does not use the names: a root of any name counts. Langfuse
marks the root of each trace (`isRootObservation`). A root of the type of one step (a model call, a tool
call, a retrieval) is not a request: a check or an evaluation ran that step by itself. A run of the test set
(`make testset`) also sends requests, so its traces count too.

In [ ]:
roots = [o for o in observations("core,basic,metrics", isRootObservation=True) if o["type"] not in STEPS]
generations = observations("core,basic,usage", type="GENERATION")
cost = pd.DataFrame(generations, columns=["traceId", "totalCost"]).groupby("traceId")["totalCost"].sum()
requests = pd.DataFrame(
    [{"trace_id": o["traceId"], "day": o["startTime"][:10], "name": o["name"], "latency_s": o["latency"]} for o in roots],
    columns=["trace_id", "day", "name", "latency_s"],
)
requests["cost_usd"] = requests["trace_id"].map(cost).fillna(0.0)
daily = requests.groupby("day").agg(
    requests=("trace_id", "count"),
    latency_s_p95=("latency_s", lambda values: percentile(list(values), 0.95)),
    cost_usd_mean=("cost_usd", "mean"),
)
print(f"{len(requests)} requests in the last {DAYS} days")
print("Requests by the name of the root observation:", requests["name"].value_counts().to_dict())
if len(requests):
    print(f"95th-percentile latency: {percentile(list(requests['latency_s']), 0.95):.2f} s")
    print(f"Mean cost of a request: ${requests['cost_usd'].mean():.5f}")
daily

In [ ]:
if len(daily):
    daily[["latency_s_p95"]].plot(title="95th-percentile latency of a request (s)", marker="o", legend=False)
    daily[["cost_usd_mean"]].plot(title="Mean cost of a request (USD)", marker="o", legend=False)

## The feedback of the users

The score `user-feedback` of week 26: 1 for a good answer, 0 for a bad one. The share of good answers of
each day.

In [ ]:
scores = list(pages("/api/public/v3/scores", name="user-feedback", fromTimestamp=SINCE.isoformat(),
                    fields="core,details,subject"))
feedback = pd.DataFrame(
    [{"day": s["timestamp"][:10], "good": float(s["value"]), "comment": s.get("comment")} for s in scores],
    columns=["day", "good", "comment"],
)
feedback_daily = feedback.groupby("day").agg(answers=("good", "count"), share_good=("good", "mean"))
print(f"{len(feedback)} ratings in the last {DAYS} days")
feedback_daily

## The evaluation scores over time

One point for each run that a result file holds, at the time of the run:

| File | Score |
| --- | --- |
| `testset-<time>.jsonl` (Phase 3) | the share of cases with `code_pass` |
| `compare-<loop>-<provider>.json` (Phase 4) | `score` |
| `agent-tasks.json` (Phase 4) | `result_score` and `path_score` |
| `attacks-<label>.json` (Phase 5) | `pass_rate` |

Most commands write over their file, so these files keep only the last run. `make testset` writes a new
file for each run: keep these files to see the trend.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


points = []
for path in sorted(RESULTS.glob("testset-*.jsonl")):
    stamp = re.search(r"(\d{8}-\d{6})", path.name)
    rows = read_jsonl(path)
    if stamp and rows:
        time = datetime.strptime(stamp.group(1), "%Y%m%d-%H%M%S").replace(tzinfo=timezone.utc)
        points.append((time, "test set: code_pass", sum(bool(r.get("code_pass")) for r in rows) / len(rows), path.name))
for path in sorted(RESULTS.glob("compare-*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    points.append((data["created"], f"{path.stem}: score", data["score"], path.name))
if (RESULTS / "agent-tasks.json").exists():
    data = json.loads((RESULTS / "agent-tasks.json").read_text(encoding="utf-8"))
    points += [(data["created"], f"agent tasks: {key}", data[key], "agent-tasks.json") for key in ("result_score", "path_score")]
for path in sorted(RESULTS.glob("attacks-*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    points.append((data["created"], f"attacks ({data.get('model')}): pass_rate", data["pass_rate"], path.name))

evaluation = pd.DataFrame(points, columns=["time", "score", "value", "file"])
evaluation["time"] = pd.to_datetime(evaluation["time"], utc=True, format="ISO8601")
evaluation = evaluation.sort_values("time").reset_index(drop=True)
evaluation

In [ ]:
if len(evaluation):
    chart = evaluation.pivot_table(index="time", columns="score", values="value")
    chart.plot(title="Evaluation scores over time", marker="o", ylim=(0, 1.05)).legend(fontsize="small")